# Gemini API - Prompt Evals
Notebook de exploracion para el SDK de Google Gemini, replica de `001_prompt_evals.ipynb` (Anthropic) usando la API de Gemini, siguiendo la misma estructura que `002_gemini_requests.ipynb`.

In [20]:
# Load env variables and create client
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client()
model = "gemini-3.5-flash"

In [21]:
# Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "parts": [{"text": text}]})


def add_assistant_message(messages, text):
    messages.append({"role": "model", "parts": [{"text": text}]})


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "contents": messages,
    }

    config_kwargs = {}
    if system:
        config_kwargs["system_instruction"] = system
    if stop_sequences:
        config_kwargs["stop_sequences"] = stop_sequences

    if config_kwargs:
        params["config"] = types.GenerateContentConfig(**config_kwargs)

    response = client.models.generate_content(**params)
    return response.text

In [22]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [23]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [24]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [33]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-30 key strengths
    - "weaknesses": An array of 1-30 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-100
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [26]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade.get("score", 0)
    reasoning = model_grade.get("reasoning", "")
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [30]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score across all test cases: {average_score}")

    return results

In [31]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score across all test cases: 1


In [32]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here is the Python function using the `boto3` library to list all S3 buckets in the AWS account that have bucket versioning enabled.\n\nIt uses `list_buckets` to get all bucket names, and then checks the versioning status of each bucket using `get_bucket_versioning`.\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef get_versioned_buckets() -> list[str]:\n    \"\"\"\n    Lists the names of all S3 buckets in the current AWS account \n    that have bucket versioning enabled.\n    \n    Returns:\n        list: A list of bucket names (strings) with versioning enabled.\n    \"\"\"\n    s3_client = boto3.client('s3')\n    versioned_buckets = []\n\n    try:\n        # Retrieve all buckets in the account\n        response = s3_client.list_buckets()\n    except ClientError as e:\n        print(f\"Error listing buckets: {e}\")\n        return []\n\n    for bucket in response.get('Buckets', []):\n        bucket_name = bucket['Name']\n        try:\n